# 第 10 集：NumPy 复制与视图

    > 对应《Numpy & Pandas 数据处理教程》课程。Notebook 按“概念 → 示例 → 观察结果”的顺序整理，建议逐格运行，并尝试修改示例数据。

    ## 本节目标


- 区分赋值别名、视图和独立副本
- 理解基础切片为什么会影响原数组
- 会用 `copy()` 隔离修改


## 1. 普通赋值只是增加别名

`b = a` 不会复制数组。两个变量指向同一个对象，因此通过 `b` 修改内容，`a` 也会变化。


In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
b = a
b[0] = 100

a, b, a is b


## 2. `view()` 创建新对象但共享数据

视图对象和原数组不是同一个 Python 对象，但底层数据仍然共享。改动元素仍会互相影响。


In [ ]:
original = np.array([10, 20, 30, 40])
viewed = original.view()
viewed[1] = 999

original, viewed, original is viewed, np.shares_memory(original, viewed)


## 3. 基础切片通常返回视图

这是 NumPy 与 Python 列表很容易混淆的地方：数组切片通常共享原数据，以避免不必要的大量复制。


In [ ]:
source = np.arange(6)
slice_view = source[1:4]
slice_view[:] = -1

source, slice_view, np.shares_memory(source, slice_view)


## 4. `copy()` 创建独立数据

当你需要安全修改中间结果、不希望影响原数组时，显式调用 `copy()`。


In [ ]:
source = np.arange(6)
independent = source[1:4].copy()
independent[:] = 99

source, independent, np.shares_memory(source, independent)


## 5. 花式索引与布尔索引通常会复制

和基础切片不同，整数数组索引、布尔索引通常返回副本。不要只凭外观判断，必要时用 `np.shares_memory()` 验证。


In [ ]:
data = np.arange(8)
fancy = data[[1, 3, 5]]
boolean = data[data % 2 == 0]

np.shares_memory(data, fancy), np.shares_memory(data, boolean)


## 三种关系对照

| 写法 | 新对象 | 共享底层数据 | 修改可能影响原数组 |
|---|---:|---:|---:|
| `b = a` | 否 | 是 | 是 |
| `b = a.view()` / 基础切片 | 是 | 是 | 是 |
| `b = a.copy()` | 是 | 否 | 否 |


## 本节小结

读取数据时视图高效，修改数据时副本更安全。只要后续要改内容，就先判断是否需要 `copy()`。
